|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Continuous batching<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: write the iteration-level scheduler<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(1)

Write the scheduler.

One function, one loop, and a hook for the admission policy so you can change
your mind about it in Exercise 4. This is stage 05 of the ladder with the GPU
replaced by a counter, which is the right way to get a scheduler correct
before you make it fast.

In [ ]:
### run this cell

N = 3000
B = 64
lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=N).astype(int) + 1
capacity = B / lengths.mean()
arrive   = np.cumsum(rng.exponential(1/(0.7*capacity), size=N))

print(f'{N} requests, mean {lengths.mean():.0f} tokens, {B} slots')

# Exercise 1: the step loop

Four things happen, in this order, every step:

1. arrivals join the waiting queue
2. free slots are filled from it
3. every running sequence emits one token
4. anything that just finished leaves its slot immediately

Step 4 is the whole idea. A static batcher does it at the end of the batch.

In [ ]:
def run(arrive, lengths, B, pick):
  """pick(waiting, lengths) -> index INTO `waiting` of the one to admit."""
  done, started = np.zeros(len(lengths)), np.zeros(len(lengths))
  t, nxt, running, waiting = 0.0, 0, {}, []
  busy = []

  while nxt < len(lengths) or waiting or running:

    # 1. everything that has arrived by now joins the waiting queue
    while nxt < len(lengths) and arrive[nxt] <= t:
      

    # 2. fill every free slot from the waiting queue, using `pick`
    while len(running) < B and waiting:
      r = 
      

    if not running:
      t = arrive[nxt]; continue     # nothing to do, skip ahead

    busy.append(len(running))

    # 3. ONE decode step. Every running sequence emits one token.
    t += 1.0
    for r in list(running):
      
      # 4. anything that just finished leaves its slot NOW,
      #    not at the end of some batch
      

  return done, started, np.array(busy)

fcfs = lambda waiting, lengths: 0     # admit the oldest request
done, started, busy = run(arrive, lengths, B, fcfs)
print(f'finished at step {done.max():,.0f}')

# Exercise 2: prove it is not lying

A scheduler that drops requests or hands out extra tokens will still produce
a plausible-looking throughput number. Check the invariants.

In [ ]:
# every request finished, got exactly its own number of tokens, never
# started before it arrived, and the slot limit was never exceeded
assert (done > 0).all(), 'some request never finished'
assert , 'wrong number of tokens somewhere'
assert , 'a request started before it arrived'
assert , 'more sequences running than there are slots'
print('all checks passed')
print(f'mean occupancy {busy.mean():.1f} of {B} slots ({100*busy.mean()/B:.0f}%)')

# Exercise 3: against static batching

In [ ]:
def static_batching(arrive, lengths, B):
  out = np.zeros(len(lengths)); t = 0.0; i = 0
  while i < len(lengths):
    b = np.arange(i, min(i+B, len(lengths)))
    t = max(t, arrive[b[-1]]); s = lengths[b].max()
    out[b] = t + s; t += s; i += B
  return out

ds = static_batching(arrive, lengths, B)
lat_s, lat_c = , 

print(f"{'':<12} {'makespan':>10} {'p50 lat':>9} {'p99 lat':>9}")
print(f"{'static':<12} {ds.max():>10,.0f} {np.median(lat_s):>9,.0f} {np.percentile(lat_s,99):>9,.0f}")
print(f"{'continuous':<12} {done.max():>10,.0f} {np.median(lat_c):>9,.0f} {np.percentile(lat_c,99):>9,.0f}")
print(f'\nthroughput {ds.max()/done.max():.2f}x, p99 latency {np.percentile(lat_s,99)/np.percentile(lat_c,99):.0f}x better')

# Exercise 4: change the admission policy

First-come-first-served is one choice. Try admitting the shortest request
instead, and look at what it does to each part of the distribution rather
than at the average.

One thing first: at 70% load your waiting queue is empty almost every step,
so `pick` never gets a choice and every policy scores the same. Overload the
server to 120% and the question becomes real. A scheduler is only a scheduler
when there is something to schedule.

In [ ]:
# admit the request with the fewest tokens left to generate.
# `pick` is given the waiting list and the lengths array.
sjf = lambda waiting, lengths: 

d2, _, _ = run(busy_arrive, lengths, B, sjf)

print(f"{'policy':<22} {'makespan':>10} {'p50':>8} {'p99':>9} {'worst':>9}")
for nm, d in (('first come first served', base), ('shortest job first', d2)):
  L = d - busy_arrive
  print(f'{nm:<22} {d.max():>10,.0f} {np.median(L):>8,.0f} {np.percentile(L,99):>9,.0f} {L.max():>9,.0f}')

### Before you open the solution

1. Continuous batching beat static on throughput **and** on latency.
   Those usually trade against each other. Why did this one not?
2. Compare the `p50` and the `worst` columns between the two policies.
   Who wins under shortest-job-first, and who pays for it?
3. Look at what your `sjf` function had to read to make its decision.
   Could a real server have known that? What would it have to do
   instead?